In this notebook, I will provide an overview of the dataset, perform cleaning and formatting based on the assumption checks conducted in 02_assumptions.ipynb, and saving it into a parquet format prior to performing the exploratory data analysis (EDA).

In [ ]:
import pandas as pd
import os
import seaborn as sns 
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)

In [ ]:
df_raw = pd.read_csv("../data/raw/ncr_ride_bookings.csv")


In [ ]:
df_raw.head()

In [ ]:
df_raw.info(memory_usage = "deep")

I have a list of to-do's from the previous notebook:
 
1. Convert names into snake case
2. Map target variable 
3. Remove data leakage
4. Format Booking ID and Customer ID
5. Recast columns 


## Convert names into snake case

In [ ]:
import re 
df_columns_clean = df_raw.copy()
def to_snake_case(df):
    new_columns = []
    for col in df.columns:
        col = col.strip()
        col = re.sub(r'[\s\-]+', '_', col)
        col = re.sub(r'[^\w_]', '', col)
        col = col.lower()
        new_columns.append(col)
    
    df.columns = new_columns
    return df

df_columns_clean = to_snake_case(df_columns_clean)
print(df_columns_clean.columns.tolist())

## Map target variable

In [ ]:
df_target_mapped = df_columns_clean.copy()

booking_mapping = {
    "Completed": 0,
    "Cancelled by Driver": 1,
    "No Driver Found": 1,
    "Cancelled by Customer": 1,
    "Incomplete": 0
}

df_target_mapped['is_cancelled'] = df_target_mapped['booking_status'].map(booking_mapping)
del df_target_mapped["booking_status"]
df_target_mapped['is_cancelled'].value_counts()

## Remove data leakage

Leaking columns: 
- Cancelled Rides by Driver
- Cancelled Rides by Customer
- Reason for cancelling by Customer
- Driver Cancellation Reason
- Incomplete Rides
- Incomplete Rides Reason
- Driver Ratings
- Customer Rating

In [ ]:
df_leak_clean = df_target_mapped.copy()

df_leak_clean.drop(columns = ["cancelled_rides_by_customer",
                                "reason_for_cancelling_by_customer",
                                "cancelled_rides_by_driver",
                                "driver_cancellation_reason",
                                "incomplete_rides",
                                "incomplete_rides_reason",
                                "driver_ratings",
                                "customer_rating"], inplace = True)

df_leak_clean.shape[0], df_leak_clean.shape[1]

## Format Booking ID and Customer ID

In [ ]:
df_clean_quotes = df_leak_clean.copy()

cols_with_quotes = ["booking_id", 
                    "customer_id"]

for col in cols_with_quotes:
    df_clean_quotes[col] = df_clean_quotes[col].str.strip('"')

df_clean_quotes[["booking_id", 
                    "customer_id"]]

## Recast columns

In [ ]:
df_typed = df_clean_quotes.copy()

df_typed.info()

### Temporal features

Converting column `date` and `time` would add unnecessary complexity so I will merge them into datetime later. 

### Categorical
1. Columns with "object" data type and no nulls are directly turned into Category
   
2. Columns with "object" data type and nulls are turned into Category based on threshold. Threshold is set to 0.5, that means less than 50% of values are unique. If threshold is less that this value, the column will be turned into string[pyarrow] type. 

In [ ]:
cat_ambiguous = ["booking_id", 
               "customer_id" ]
cat_threshold = 0.5

for col in cat_ambiguous:
    if col in df_typed.columns:
        num_unique = df_typed[col].nunique(dropna = False)
        ratio_unique = num_unique/len(df_typed)

        if ratio_unique <= cat_threshold:
            df_typed[col] = df_typed[col].astype("category")
            print(f"- Column {col} has been transformed into Category")
            print(f"Ratio of the column: {ratio_unique:.4%}")
        else:
            df_typed[col] = df_typed[col].astype("string[pyarrow]")
            print(f"- Column {col} has been transformed into string[pyarrow]")
            print(f"Ratio of the column: {ratio_unique:.4%}")


In [ ]:
cat_columns = ["vehicle_type", 
               "pickup_location", 
               "drop_location",
               "payment_method"]

for col in cat_columns:
    if col in df_typed.columns:
        df_typed[col] = df_typed[col].astype("category")
        print(f"- Column {col} has been transformed into Category")

### Numerical

i usually perform type optimization since I'm working on my personal computer. In this case, since the dataset is not very big, I will only transform the numerical data into float32

In [ ]:
numerical_columns = ["avg_vtat", 
                 "avg_ctat", 
                 "booking_value",
                 "ride_distance",
                 "is_cancelled"]

for col in numerical_columns:
    if col in df_typed.columns:
        df_typed[col] = df_typed[col].astype("float32")
        print(f"- Column {col} has been transformed into float32")


In [ ]:
df_typed.to_parquet("../data/bronze/clean_dataset.parquet",  engine='fastparquet', index=False)

## Train/test split

The dataset goes from the 1st of January 2024 to the 30th December 2024, I will split it following 75% train, 15% val, 10% test.

In [ ]:
df_typed = df_typed.sort_values("date")

train_end = "2024-09-30"  
val_end = "2024-11-30"

train_df = df_typed[df_typed["date"] <= train_end]
val_df   = df_typed[(df_typed["date"] > train_end) & (df_typed["date"] <= val_end)]
test_df  = df_typed[df_typed["date"] > val_end]

In [ ]:
print(f" Train: {len(train_df)}")
print(f" Validation : {len(val_df)}")
print(f" Test: {len(test_df)}")

## Save dataset splits

In [ ]:
output_dir = "../data/bronze"
os.makedirs(output_dir, exist_ok=True)  
df_typed.to_parquet(os.path.join(output_dir, "dataset.parquet"), index=False)
train_df.to_parquet(os.path.join(output_dir, "train.parquet"), index=False)
val_df.to_parquet(os.path.join(output_dir, "val.parquet"), index=False)
test_df.to_parquet(os.path.join(output_dir, "test.parquet"), index=False)